In [ ]:
import sys

In [ ]:
sys.path.append("..")
from wavelet_transform.utils.save_plots import save_figure

from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.utils.kmeans_results as results

In [ ]:
savefigs_flag = True

Bubble dynamics results

In [ ]:
# # Get bubble dynamics results
# frequency = 2e6


# exp_ids = [f"{R_init}_{frequency}_{pac}" for pac, R_init in [ (1e6, 15e-6),(1e6, 1e-6)]]

# # R_init =1e-6
# # pac = 1e6
# # exp_ids = [f"{R_init}_{frequency}_{pac}"]
# # print(exp_ids)
# exp_results = [results.get_data(result_id) for result_id in exp_ids]

# exp_labels = ["Stable", "Transient"]  # Legend for final plot

# label_maps = [
#     {0: "measurement noise", 1: "fundamental"},
#     {0:  "transient", 1:"fundamental and harmonics"}
# ]

# # label_maps = [
# #     {0: "broadband", 1: "measurement noise", 2: "fundamental"},
# #     {0: "broadband", 1: "fundamental", 2: "measurement noise", 3:"??"}
# # ]



Results for other datasets

In [ ]:
# # Get results from experiment number
# exp_number = 9  # 9, 15, 55
# result_id = str(exp_number)
# frequency = 2e6

# exp_ids = [9, 15, 55]
# exp_results = [results.get_data(result_id) for result_id in exp_ids]
# label_maps = [
#     {0: "broadband", 1: "measurement", 2:"fundamental"},
#     {0: "broadband", 1: "harmonic", 2:"measurement", 3: "fundamental", 4:"transient"},
#     {0: "measurement", 1: "broadband", 2: "fundamental"}
# ]

In [ ]:
# Get PCD results
voltage = 80
pcd_number = 1
no_clusters = 2
result_id = f"{voltage}_{pcd_number}_{no_clusters}"
frequency = 1.1e6
exp_results = results.get_data(result_id)

label_map = {1:"noise", 0:"harm/bb", 2:"fund.", 3:"bb"}
label_map = {1:"noise", 0:"harm/bb", 2:"bb", 3:"fund.", 4:"sub/bb"}
label_map = {1: "periodic", 0:"transient"}

In [ ]:
# Keep track of all results for combined plot in next cell

required_info = ["data", "tr_obj", "centers", "labels", "sampling rate", "freq_threshold"]
# Plot separate clusterig results
label_map = label_map
# unpack results
data_train, tr_obj, centers, labels, sampling_rate, freq_threshold = (
    tuple(exp_results[i] for i in required_info)
)
classifying_metrics = data_train.columns.to_list()
print(label_map)
fig = plot.plot_clustering(
    data_train, centers, labels, 
    metrics_to_plot=classifying_metrics,
    title=None, 
    label_map=label_map
)

if savefigs_flag == True:
    save_figure(fig, f"clustering_{result_id}")

In [ ]:
# Plot centroid metrics
fig = plot.plot_centers(classifying_metrics, centers,label_map=label_map)
if savefigs_flag == True:
    save_figure(fig, f"sig_components_metrics_{result_id}")

- change legend to stable and transient
- add error bars for multiple repetitions of the same experiment

In [ ]:
# Set clusters to None to plot all of them, otherwise, plot a list of int
fig = plot.plot_fft_by_group(
    tr_obj, labels, sampling_rate=sampling_rate,
    clusters=None, xlim=10, fundamental_freq=frequency,
    freq_threshold=freq_threshold, label_map=label_map
)
if savefigs_flag == True:
    save_figure(fig, f"fft_rec_centers_{result_id}")

fig = plot.plot_reconstructions_by_group(
    tr_obj, labels, t_plot=sampling_rate,
    freq_threshold=freq_threshold, label_map=label_map
)
if savefigs_flag == True:
    save_figure(fig, f"sig_rec_centers_{result_id}")


In [ ]:
tr_obj.calc_frequencies(sampling_rate)
vmin = -9
vmax = 2
fig = plot.heisenberg(
    tr_obj, 1/sampling_rate*len(tr_obj.data), max_freq=.1, padding="repeat",
    vmin=vmin, vmax=vmax
)
if savefigs_flag is True:
    save_figure(fig, f"heisenberg_{result_id}")

In [ ]:
freq_threshold

In [ ]:
from wavelet_transform.utils.statistical_features import kurtosis

reconstructions = plot.get_reconstructions_by_group(tr_obj, labels, freq_threshold)
for i, rec in enumerate(reconstructions):
    print(f"kurtosis of reconstructed cluster {i}: {kurtosis(rec)}")